# Multimodal Human Activity Recognition on MMAct (Kaggle GPU Pipeline)
## Fast, Reliable, Complete 50-Epoch Benchmark: Custom CNN, Tuned CNN, MobileNetV2 Staged Fine-Tuning, Sensor-Only, and Multimodal Fusion

This notebook executes the complete Multimodal Human Activity Recognition (HAR) pipeline on the **MMAct dataset** across structured, bounded sections:
- **Section 1**: Environment & Dependencies Setup (strictly preserving Kaggle's binary C ABI stack)
- **Section 2**: Preflight Hardware & GPU Diagnostics (CUDA device, memory headroom, CPU, RAM, disk, fail-loud GPU check)
- **Section 3**: Repository Checkout from `master` branch
- **Section 4**: Fast, Idempotent Dataset Discovery & Canonical Staging (shallow inspection, zero duplicate links, preflight audit report)
- **Section 5**: Label Mapping & Pre-Flight Modality Audit (Zero-Leakage Split Verification across 10 target classes)
- **Section 6**: Video & Sensor Preprocessing Cache Warm-Up (`/kaggle/working/har_mmact_cache`)
- **Section 7**: Memory-Safe Calibration & OOM Warm-Up
- **Section 8**: Experiment 1: Improved Custom CNN Baseline (50 epochs, AMP, spatial dropout, weight decay, skip-if-completed)
- **Section 9**: Experiment 2: Hyperparameter Tuning (train/val only) & Tuned CNN (50 epochs)
- **Section 10**: Experiment 3: MobileNetV2 Transfer Learning with Staged Fine-Tuning (50 epochs, AMP)
- **Section 11**: Experiments 4 & 5: Sensor-Only InceptionTime & Multimodal Late Fusion (50 epochs each)
- **Section 12**: Standalone Evaluation on Held-Out Test Set across all 5 models
- **Section 13**: Comprehensive Publication Visualizations & Galleries (annotated curves, ROC, PR, confusion matrices)
- **Section 14**: Sample Multimodal Inference Demonstration
- **Section 15**: Consolidated Summary Table, Manifest, Runtime Report & 1-Click Packaging (`har_mmact_results.zip`)
- **Section 16**: (Optional) Extended Benchmarks (100 and 150 Epochs for extended quota)

All experiments evaluate on the **10 verified target activity classes**:
`crouching`, `jumping`, `kicking`, `pointing`, `running`, `sitting`, `standing`, `using_phone`, `walking`, `waving_hand`.

---
### Kaggle Setup Instructions:
1. **Enable GPU Accelerator:** Under **Session Options** -> **Accelerator**, select **GPU T4 x2** or **GPU P100**.
2. **Enable Internet Access:** Under **Session Options** -> **Internet**, toggle to **On** (required for GitHub repo checkout).
3. **Attach MMAct Dataset:** Click **+ Add Input** and attach your MMAct dataset (e.g. `mmact-trimmed`).

## Section 1: Environment & Dependencies Setup
Installs auxiliary dependencies while strictly preserving Kaggle's preinstalled, binary-compatible C ABI scientific stack (NumPy, Pandas, SciPy, PyTorch, Torchvision, Scikit-Learn).

In [ ]:
# Section 1: Dependencies Setup with Kaggle C ABI Protection
import sys
import os
import subprocess
from pathlib import Path

PROJECT_ROOT = Path('/kaggle/working/HAR-MMAct')
CORE_C_PACKAGES = {'numpy', 'pandas', 'scipy', 'torch', 'torchvision', 'opencv-python', 'pillow', 'scikit-learn', 'matplotlib', 'seaborn', 'pytest'}

kaggle_req = PROJECT_ROOT / 'requirements-kaggle.txt'
if kaggle_req.exists():
    print(f'Installing auxiliary dependencies from {kaggle_req.name}...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--prefer-binary', '--no-deps', '-r', str(kaggle_req)], check=False)
else:
    # Install standard light requirements without overriding core C packages
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tabulate', 'psutil'], check=False)

# Post-install C ABI Verification
print('\n' + '=' * 70)
print('POST-INSTALL ABI & IMPORT VERIFICATION')
print('=' * 70)
abi_failed = False
abi_errors = []

try:
    import numpy as np
    print(f'  [PASS] Python      : {sys.version.split()[0]}')
    print(f'  [PASS] NumPy       : {np.__version__}')
    _arr = np.eye(2, dtype=np.float64)
    _ = np.linalg.pinv(_arr)
except Exception as e:
    abi_failed = True
    abi_errors.append(f'NumPy C-extension check failed: {e}')

try:
    import pandas as pd
    print(f'  [PASS] Pandas      : {pd.__version__}')
    _df = pd.DataFrame({'a': [1.0, 2.0], 'b': [3.0, 4.0]})
    _ = _df.mean()
except Exception as e:
    abi_failed = True
    abi_errors.append(f'Pandas C-extension check failed: {e}')

try:
    import scipy.signal
    print(f'  [PASS] SciPy       : {scipy.__version__}')
    _ = scipy.signal.resample([1.0, 2.0, 3.0, 4.0], 8)
except Exception as e:
    abi_failed = True
    abi_errors.append(f'SciPy C-extension check failed: {e}')

try:
    import cv2
    print(f'  [PASS] OpenCV      : {cv2.__version__}')
except Exception as e:
    abi_failed = True
    abi_errors.append(f'OpenCV check failed: {e}')

try:
    import torch
    print(f'  [PASS] PyTorch     : {torch.__version__} (CUDA: {torch.cuda.is_available()})')
    _t = torch.randn(2, 2)
    _ = _t.numpy()
except Exception as e:
    abi_failed = True
    abi_errors.append(f'PyTorch <-> NumPy C ABI bridge check failed: {e}')

try:
    import sklearn
    print(f'  [PASS] Scikit-Learn: {sklearn.__version__}')
except Exception as e:
    abi_failed = True
    abi_errors.append(f'Scikit-Learn check failed: {e}')

if abi_failed:
    print('\n' + '!' * 70)
    print('CRITICAL ABI BINARY INCOMPATIBILITY DETECTED:')
    for err in abi_errors:
        print(f'  - {err}')
    print('\nREMEDY: Restart Kaggle Session from top menu, do NOT run pip install numpy manually.')
    print('!' * 70)
    raise RuntimeError('C ABI binary incompatibility detected:\n' + '\n'.join(abi_errors))

print('=' * 70)
print('>>> C ABI & IMPORT VERIFICATION PASSED: Preinstalled scientific stack is healthy.')
print('=' * 70)


## Section 2: Preflight Hardware & GPU Diagnostics
Audits GPU accelerator, CUDA device capability, VRAM headroom, CPU threads, host RAM, and working disk space. Enforces fail-loud check if GPU accelerator is disabled.

In [ ]:
# Section 2: Preflight Hardware & GPU Diagnostics
import os
import shutil
import psutil
import torch

!nvidia-smi

print('=' * 78)
print(' PREFLIGHT HARDWARE & GPU DIAGNOSTICS')
print('=' * 78)

# 1. CUDA & GPU Verification
cuda_ok = torch.cuda.is_available()
print(f'PyTorch Version       : {torch.__version__}')
print(f'CUDA Available        : {cuda_ok}')

if not cuda_ok:
    error_msg = (
        'CRITICAL FAILURE: CUDA GPU is not available!\n'
        'This 50-epoch benchmark requires a GPU accelerator.\n'
        'ACTION REQUIRED: In Kaggle right-hand panel -> Session Options -> Accelerator, '
        'select GPU T4 x2 or GPU P100, then restart the session.'
    )
    print('!' * 78)
    print(error_msg)
    print('!' * 78)
    raise RuntimeError(error_msg)

gpu_count = torch.cuda.device_count()
gpu_name = torch.cuda.get_device_name(0)
compute_cap = torch.cuda.get_device_capability(0)
props = torch.cuda.get_device_properties(0)
total_vram_gb = props.total_memory / (1024**3)
alloc_vram_mb = torch.cuda.memory_allocated(0) / (1024**2)
res_vram_mb = torch.cuda.memory_reserved(0) / (1024**2)

print(f'GPU Device Name       : {gpu_name} (x{gpu_count} detected)')
print(f'Compute Capability    : {compute_cap[0]}.{compute_cap[1]}')
print(f'Total VRAM Available  : {total_vram_gb:.2f} GB')
print(f'Initial Allocated VRAM: {alloc_vram_mb:.2f} MB')
print(f'Initial Reserved VRAM : {res_vram_mb:.2f} MB')

# 2. Host System Hardware
cpu_count = os.cpu_count() or 1
vm = psutil.virtual_memory()
ram_total_gb = vm.total / (1024**3)
ram_avail_gb = vm.available / (1024**3)

working_disk = shutil.disk_usage('/kaggle/working')
disk_free_gb = working_disk.free / (1024**3)
disk_total_gb = working_disk.total / (1024**3)

print(f'CPU Logical Cores     : {cpu_count}')
print(f'System RAM Total      : {ram_total_gb:.2f} GB (Available: {ram_avail_gb:.2f} GB)')
print(f'Working Disk Space    : {disk_free_gb:.2f} GB free of {disk_total_gb:.2f} GB')

# 3. Runtime Training Settings
print('\nRuntime Acceleration Configuration:')
print('  - Batch Size          : 8 (calibrated for 16-frame 112x112 video + 9-channel sensor)')
print('  - DataLoader Workers  : 2 (non-blocking pin_memory)')
print('  - Automatic Mixed Prec: ENABLED (torch.amp.autocast, float16)')
print('  - cuDNN Benchmark     : ENABLED (torch.backends.cudnn.benchmark = True)')
print('=' * 78)


## Section 3: Repository Checkout from master Branch
Clones or pulls the latest verified code from the `master` branch of `https://github.com/CHENNAMSETTYVIKRAM/HAR-MMAct.git`, configures Python `sys.path`, and verifies project modules.

In [ ]:
# Section 3: Repository Checkout from master Branch
import sys
import os
import subprocess
from pathlib import Path

PROJECT_ROOT = Path('/kaggle/working/HAR-MMAct')
REPO_URL = 'https://github.com/CHENNAMSETTYVIKRAM/HAR-MMAct.git'
BRANCH = 'master'

# 1. Clone or update repository into PROJECT_ROOT explicitly on 'master' branch
if not PROJECT_ROOT.exists():
    print(f'Cloning {REPO_URL} (branch: {BRANCH}) into {PROJECT_ROOT}...')
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    print(f'Repository already present at {PROJECT_ROOT}. Fetching and checking out {BRANCH}...')
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', BRANCH], check=False)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', BRANCH], check=False)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', 'origin', BRANCH], check=False)

# 2. Print active branch and commit hash for diagnostics
try:
    branch_out = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', '--abbrev-ref', 'HEAD'], text=True).strip()
    commit_out = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
    print(f'Active Git Branch : {branch_out}')
    print(f'Active Git Commit : {commit_out}')
except Exception as e:
    print(f'[Warning] Could not inspect git metadata: {e}')
    branch_out, commit_out = 'unknown', 'unknown'

# 3. Verify expected files exist on master
req_files = [
    PROJECT_ROOT / 'src' / 'data' / 'ingestion.py',
    PROJECT_ROOT / 'src' / 'data' / 'indexer.py',
    PROJECT_ROOT / 'src' / 'data' / 'dataset.py',
    PROJECT_ROOT / 'src' / 'data' / 'stager.py',
    PROJECT_ROOT / 'scripts' / 'train.py',
    PROJECT_ROOT / 'scripts' / 'tune.py',
    PROJECT_ROOT / 'scripts' / 'evaluate.py',
]
missing_files = [str(f.relative_to(PROJECT_ROOT)) for f in req_files if not f.exists()]
if missing_files:
    raise FileNotFoundError(f'CRITICAL: Missing required files on branch {branch_out}: {missing_files}')

print('SUCCESS: Verified presence of core modules on master branch.')

# 4. Configure import path before importing project modules
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print(f'PROJECT_ROOT configured : {PROJECT_ROOT}')
print(f'sys.path[0] set to      : {sys.path[0]}')

# 5. Verify project module imports
from src.data.ingestion import DatasetIngestionPipeline
from src.data.indexer import MMActIndexer
from src.data.dataset import MMActMultimodalDataset
from src.data.stager import FastMMActStager, StagingReport
from src.models.fusion_model import MultimodalHARModel
from src.utils.plots import plot_cross_model_comparison, plot_modality_ablation
print('SUCCESS: Verified project imports (ingestion, indexer, dataset, stager, models, plots).')


## Section 4: Fast, Idempotent Dataset Discovery & Canonical Staging
Discovers the attached dataset using shallow directory inspection (bounded depth $\le$ 4) across `/kaggle/input`, avoids unbounded recursive file walking over mounted filesystems, and idempotently stages a canonical dataset hierarchy at `/kaggle/working/HAR-MMAct/data/genuine_mmact/`.

**Guarantees:**
- Idempotent: Re-running skips valid existing links/files; creates zero duplicate or nested links.
- Never deletes the canonical directory.
- Cycle & symlink loop protected.
- Preflight audit report prints selected root, modalities, counts, duration, and reuse status.

In [ ]:
# Section 4: Fast, Idempotent Dataset Discovery & Canonical Staging
from pathlib import Path
from src.data.stager import FastMMActStager

kaggle_input = Path('/kaggle/input')
canonical_data_dir = PROJECT_ROOT / 'data' / 'genuine_mmact'

# Candidate roots to inspect shallowly
input_roots = sorted([p for p in kaggle_input.iterdir() if p.is_dir()]) if kaggle_input.exists() else []
if not input_roots:
    local_data = PROJECT_ROOT / 'data'
    if local_data.exists():
        input_roots = [local_data]

print(f'Scanning {len(input_roots)} candidate input roots using shallow bounded inspection (max_depth=4)...')
stager = FastMMActStager(
    canonical_dir=canonical_data_dir,
    input_roots=input_roots,
    max_scan_depth=4,
)

staging_report = stager.discover_and_stage()
print(staging_report.summary())

assert staging_report.is_valid, (
    f'CRITICAL: Dataset discovery & staging failed! Missing required items: {staging_report.missing_required_paths}\n'
    f'Please ensure the MMAct dataset is attached under /kaggle/input.'
)


## Section 5: Label Mapping & Pre-Flight Modality Audit (Zero-Leakage Split)
Verifies video frame decoding, sensor CSV parsing, index pairing, and enforces strict subject-independent split isolation (train/val/test) across the 10 target activity classes.

In [ ]:
# Section 5: Label Mapping & Pre-Flight Modality Audit
import cv2
import pandas as pd
from src.data.indexer import MMActIndexer
from src.data.splits import DatasetSplitter, verify_split_isolation
from src.config import SplitConfig, VERIFIED_TARGET_CLASSES

print('=' * 78)
print('PRE-FLIGHT VALIDATION & COMPREHENSIVE MULTI-MODALITY AUDIT')
print('=' * 78)
print(f'Repository Root : {PROJECT_ROOT}')
print(f'Canonical Root  : {canonical_data_dir.resolve()}')

# 1. File Inventory
total_videos = MMActIndexer.find_files_recursively(canonical_data_dir, {'.mp4'})
total_sensors = MMActIndexer.find_files_recursively(canonical_data_dir, {'.csv'})
total_splits = [f for f in MMActIndexer.find_files_recursively(canonical_data_dir, {'.txt'}) if any(k in f.name.lower() for k in ['cross_scene', 'cross_view', 'train', 'val', 'test'])]

print(f'CANONICAL STAGED TOTALS: {len(total_videos)} Videos | {len(total_sensors)} Sensors | {len(total_splits)} Splits')
assert len(total_videos) > 0 and len(total_sensors) > 0, 'CRITICAL: No valid MMAct data files found in canonical directory!'

# 2. Modality Readability check
sample_video = total_videos[0]
cap = cv2.VideoCapture(str(sample_video))
ret, frame = cap.read()
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()
assert ret and frame is not None, f'Failed to read frame from {sample_video}'
print(f'  [PASS] Video Readability: Decoded frame {frame.shape} from {sample_video.name} ({frame_count} frames)')

sample_sensor = total_sensors[0]
df_sample = pd.read_csv(sample_sensor)
print(f'  [PASS] Sensor Parseability: Parsed CSV {df_sample.shape} from {sample_sensor.name}')

# 3. Indexing & Target Class Coverage
indexer = MMActIndexer(dataset_root=canonical_data_dir)
report = indexer.index()
print(f'  - Paired Samples Created: {report.paired_samples_count}')
print(f'  Target Class Coverage ({len(report.class_distribution)}/{len(VERIFIED_TARGET_CLASSES)} target classes):')
for cls_name in VERIFIED_TARGET_CLASSES:
    cnt = report.class_distribution.get(cls_name, 0)
    print(f'    * {cls_name:<16}: {cnt} samples')

assert report.paired_samples_count > 0, 'Zero paired multimodal samples created!'

# 4. Subject-Independent Split Isolation Verification
splitter = DatasetSplitter(SplitConfig(split_strategy='subject'))
train_samples, val_samples, test_samples = splitter.split(report.paired_samples)
isolation = verify_split_isolation(train_samples, val_samples, test_samples, strategy='subject')
print(f'  - Splits: Train={len(train_samples)}, Val={len(val_samples)}, Test={len(test_samples)}')
for check_k, passed in isolation.items():
    print(f'      * {check_k:<28}: {"PASSED" if passed else "FAILED"}')
    assert passed, f'CRITICAL: Split leakage detected on {check_k}!'
print('=' * 78)
print('>>> PRE-FLIGHT VALIDATION PASSED: Dataset is readable, parseable, and strictly isolated.')
print('=' * 78)


## Section 6: Video & Sensor Preprocessing Cache Warm-Up
Pre-caches normalized 16-frame 112x112 video tensors and aligned 100-sample 9-axis sensor windows to `/kaggle/working/har_mmact_cache/` so all 5 subsequent experiments execute with maximum speed without repeated OpenCV decoding.

In [ ]:
# Section 6: Video & Sensor Preprocessing Cache Warm-Up
import shutil
from src.data.dataset import MMActMultimodalDataset

CACHE_DIR = Path('/kaggle/working/har_mmact_cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f'Initializing multimodal datasets with shared cache at {CACHE_DIR}...')
train_ds = MMActMultimodalDataset(train_samples, cache_dir=CACHE_DIR / 'train', enable_cache=True)
val_ds = MMActMultimodalDataset(val_samples, cache_dir=CACHE_DIR / 'val', enable_cache=True)
test_ds = MMActMultimodalDataset(test_samples, cache_dir=CACHE_DIR / 'test', enable_cache=True)

# Sequentially precompute and warm up cache
print('\n[Warming Up Training Set Cache]')
train_stats = train_ds.warm_up_cache(verbose=True)

print('\n[Warming Up Validation Set Cache]')
val_stats = val_ds.warm_up_cache(verbose=True)

print('\n[Warming Up Test Set Cache]')
test_stats = test_ds.warm_up_cache(verbose=True)

disk_free = shutil.disk_usage(str(CACHE_DIR)).free / (1024**3)
print('\n' + '=' * 78)
print(f'CACHE WARM-UP COMPLETED:')
print(f'  Train Cache Writes: {train_stats["writes"]} | Hits: {train_stats["hits"]}')
print(f'  Val   Cache Writes: {val_stats["writes"]} | Hits: {val_stats["hits"]}')
print(f'  Test  Cache Writes: {test_stats["writes"]} | Hits: {test_stats["hits"]}')
print(f'  Remaining Disk    : {disk_free:.2f} GB free')
print('=' * 78)


## Section 7: Memory-Safe Calibration & OOM Warm-Up
Executes a 1-batch calibration forward/backward pass with Automatic Mixed Precision (AMP) to verify GPU VRAM safety and establish baseline memory headroom before starting full 50-epoch training.

In [ ]:
# Section 7: Memory-Safe Calibration & OOM Warm-Up
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from src.models.fusion_model import MultimodalHARModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.cuda.reset_peak_memory_stats(0)

print('Running 1-batch GPU memory calibration test with AMP...')
calib_loader = DataLoader(train_ds, batch_size=8, shuffle=False, num_workers=0, pin_memory=True)
v_sample, s_sample, l_sample, _ = next(iter(calib_loader))

model_calib = MultimodalHARModel(
    num_classes=10,
    video_backbone='mobilenet_v2',
    sensor_backbone='inception_time',
    fusion_hidden_dim=128
).to(device)

optimizer = torch.optim.Adam(model_calib.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler('cuda')

v_gpu = v_sample.to(device, non_blocking=True)
s_gpu = s_sample.to(device, non_blocking=True)
l_gpu = l_sample.to(device, non_blocking=True)

optimizer.zero_grad(set_to_none=True)
with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
    out = model_calib(video=v_gpu, sensor=s_gpu, override_mode='multimodal')
    loss = criterion(out, l_gpu)

scaler.scale(loss).backward()
scaler.step(optimizer)
scaler.update()

peak_calib_mb = torch.cuda.max_memory_allocated(0) / (1024**2)
total_vram_mb = torch.cuda.get_device_properties(0).total_memory / (1024**2)
headroom_mb = total_vram_mb - peak_calib_mb

print(f'Peak Memory during Multimodal Calibration : {peak_calib_mb:.2f} MB')
print(f'Total VRAM Available                       : {total_vram_mb:.2f} MB')
print(f'Safe Headroom Remaining                    : {headroom_mb:.2f} MB ({headroom_mb/total_vram_mb*100:.1f}%)')

del model_calib, optimizer, criterion, scaler, v_gpu, s_gpu, l_gpu, out, loss
torch.cuda.empty_cache()
print('SUCCESS: Memory calibration passed. Safe to proceed with 50-epoch training.')


## Section 8: Experiment 1 - Improved Custom CNN Baseline (50 Epochs)
Trains the improved custom video CNN baseline for 50 epochs using Spatial Dropout2d, BatchNorm2d, Weight Decay ($10^{-3}$), Cosine Annealing, Automatic Mixed Precision, and auto-resume protection.

In [ ]:
# Section 8: Experiment 1 - Improved Custom CNN Baseline (50 Epochs)
EPOCHS = 50
out_cnn = 'outputs/baseline_cnn'
Path(out_cnn).mkdir(parents=True, exist_ok=True)

print(f'=== Starting Experiment 1: Custom CNN Baseline ({EPOCHS} Epochs) ===')
!python scripts/train.py \
    --data_root data/genuine_mmact \
    --video_backbone cnn \
    --sensor_backbone inception_time \
    --mode video_only \
    --epochs {EPOCHS} \
    --min_epochs 50 \
    --batch_size 8 \
    --lr 0.0001 \
    --weight_decay 0.001 \
    --dropout 0.35 \
    --split_strategy subject \
    --early_stopping_patience 15 \
    --checkpoint_name best_model_cnn.pth \
    --output_dir {out_cnn} \
    --cache_dir /kaggle/working/har_mmact_cache \
    --benchmark_warmup \
    --auto_resume \
    --skip_if_completed


## Section 9: Experiment 2 - Hyperparameter Tuning (Train/Val Only) & Tuned CNN (50 Epochs)
Systematically explores learning rate, weight decay, and dropout across 4 trials strictly on train/val splits. The held-out test split is NEVER touched. The best configuration is then trained for 50 epochs.

In [ ]:
# Section 9: Experiment 2 - Hyperparameter Tuning & Tuned CNN (50 Epochs)
print('=== Starting Experiment 2: Hyperparameter Tuning (Train/Val Only) ===')
!python scripts/tune.py \
    --data_root data/genuine_mmact \
    --split_strategy subject \
    --epochs_per_trial 10 \
    --batch_size 8 \
    --output_dir outputs/tuning \
    --plots_dir outputs/plots \
    --cache_dir /kaggle/working/har_mmact_cache \
    --skip_if_completed

# Load best hyperparameters discovered during tuning
import json
with open('outputs/tuning/best_hyperparameters.json') as f:
    best_cfg = json.load(f)
print(f'Optimal Hyperparameters Discovered: {best_cfg}')

out_tuned = 'outputs/tuned_cnn'
Path(out_tuned).mkdir(parents=True, exist_ok=True)

print(f'\n=== Training Tuned CNN for {EPOCHS} Epochs on Best Configuration ===')
!python scripts/train.py \
    --data_root data/genuine_mmact \
    --video_backbone cnn \
    --sensor_backbone inception_time \
    --mode video_only \
    --epochs {EPOCHS} \
    --min_epochs 50 \
    --batch_size 8 \
    --lr {best_cfg['lr']} \
    --weight_decay {best_cfg['weight_decay']} \
    --dropout {best_cfg['dropout']} \
    --split_strategy subject \
    --early_stopping_patience 15 \
    --checkpoint_name best_model_tuned_cnn.pth \
    --output_dir {out_tuned} \
    --cache_dir /kaggle/working/har_mmact_cache \
    --benchmark_warmup \
    --auto_resume \
    --skip_if_completed


## Section 10: Experiment 3 - MobileNetV2 Staged Fine-Tuning (50 Epochs)
Phase 1 (Epochs 1-15): Pretrained MobileNetV2 backbone frozen, training classification head at $10^{-4}$.
Phase 2 (Epochs 16-50): Backbone unfrozen with differential learning rate ($10^{-5}$) for gentle fine-tuning.

In [ ]:
# Section 10: Experiment 3 - MobileNetV2 Staged Fine-Tuning (50 Epochs)
out_mob = 'outputs/mobilenet_v2'
Path(out_mob).mkdir(parents=True, exist_ok=True)

print(f'=== Starting Experiment 3: MobileNetV2 Staged Fine-Tuning ({EPOCHS} Epochs) ===')
!python scripts/train.py \
    --data_root data/genuine_mmact \
    --video_backbone mobilenet_v2 \
    --sensor_backbone inception_time \
    --mode video_only \
    --epochs {EPOCHS} \
    --min_epochs 50 \
    --batch_size 8 \
    --lr 0.0001 \
    --weight_decay 0.0005 \
    --dropout 0.35 \
    --staged_finetune \
    --unfreeze_epoch 15 \
    --unfreeze_lr 0.00001 \
    --split_strategy subject \
    --early_stopping_patience 15 \
    --checkpoint_name best_model_mobilenet_v2.pth \
    --output_dir {out_mob} \
    --cache_dir /kaggle/working/har_mmact_cache \
    --benchmark_warmup \
    --auto_resume \
    --skip_if_completed


## Section 11: Experiments 4 & 5 - Sensor-Only InceptionTime & Multimodal Late Fusion (50 Epochs)
- **Experiment 4**: InceptionTime 1D convolutional network trained on 9-axis sensor time series.
- **Experiment 5**: Multimodal Late Fusion combining MobileNetV2 video representations with InceptionTime sensor features.

In [ ]:
# Section 11: Experiments 4 & 5 - Sensor-Only & Multimodal Late Fusion (50 Epochs)
out_sensor = 'outputs/sensor_only'
out_fusion = 'outputs/multimodal_fusion'
Path(out_sensor).mkdir(parents=True, exist_ok=True)
Path(out_fusion).mkdir(parents=True, exist_ok=True)

print(f'=== Starting Experiment 4: Sensor-Only Baseline ({EPOCHS} Epochs) ===')
!python scripts/train.py \
    --data_root data/genuine_mmact \
    --mode sensor_only \
    --sensor_backbone inception_time \
    --epochs {EPOCHS} \
    --min_epochs 50 \
    --batch_size 8 \
    --lr 0.0005 \
    --weight_decay 0.0001 \
    --split_strategy subject \
    --checkpoint_name best_model_sensor.pth \
    --output_dir {out_sensor} \
    --cache_dir /kaggle/working/har_mmact_cache \
    --benchmark_warmup \
    --auto_resume \
    --skip_if_completed

print(f'\n=== Starting Experiment 5: Multimodal Late Fusion ({EPOCHS} Epochs) ===')
!python scripts/train.py \
    --data_root data/genuine_mmact \
    --mode multimodal \
    --video_backbone mobilenet_v2 \
    --sensor_backbone inception_time \
    --epochs {EPOCHS} \
    --min_epochs 50 \
    --batch_size 8 \
    --lr 0.0001 \
    --weight_decay 0.0005 \
    --split_strategy subject \
    --checkpoint_name best_model_multimodal.pth \
    --output_dir {out_fusion} \
    --cache_dir /kaggle/working/har_mmact_cache \
    --benchmark_warmup \
    --auto_resume \
    --skip_if_completed


## Section 12: Standalone Evaluation on Held-Out Test Set (All 5 Models)
Evaluates all 5 saved checkpoints on the strictly isolated, held-out test split using Automatic Mixed Precision.

In [ ]:
# Section 12: Standalone Evaluation on Held-Out Test Set (All 5 Models)
import json

models_to_evaluate = [
    ('Baseline Custom CNN', 'outputs/baseline_cnn/checkpoints/best_model_cnn.pth', 'outputs/baseline_cnn'),
    ('Tuned Custom CNN', 'outputs/tuned_cnn/checkpoints/best_model_tuned_cnn.pth', 'outputs/tuned_cnn'),
    ('MobileNetV2 Fine-Tuned', 'outputs/mobilenet_v2/checkpoints/best_model_mobilenet_v2.pth', 'outputs/mobilenet_v2'),
    ('InceptionTime (Sensor-Only)', 'outputs/sensor_only/checkpoints/best_model_sensor.pth', 'outputs/sensor_only'),
    ('Multimodal Fusion (Video+Sensor)', 'outputs/multimodal_fusion/checkpoints/best_model_multimodal.pth', 'outputs/multimodal_fusion'),
]

eval_results = {}
for name, ckpt, out_d in models_to_evaluate:
    print(f'\nEvaluating {name} on held-out test split...')
    !python scripts/evaluate.py \
        --checkpoint {ckpt} \
        --data_root data/genuine_mmact \
        --output_file {out_d}/metrics.json \
        --plots_dir {out_d}/plots \
        --cache_dir /kaggle/working/har_mmact_cache \
        --skip_if_completed
    with open(f'{out_d}/metrics.json') as f:
        m_data = json.load(f)
    mode_key = 'multimodal' if 'multimodal' in m_data else 'video_only'
    if 'sensor_only' in out_d:
        mode_key = 'sensor_only'
    eval_results[name] = m_data[mode_key]


## Section 13: Comprehensive Publication Visualizations & Galleries
Generates cross-model benchmark comparisons, modality ablation bar charts, annotated training trajectories, confusion matrices, ROC curves, and PR curves.

In [ ]:
# Section 13: Comprehensive Publication Visualizations & Galleries
import json
from pathlib import Path
from IPython.display import Image, display
from src.utils.plots import (
    plot_cross_model_comparison,
    plot_modality_ablation,
)

plots_output = Path('outputs/plots')
plots_output.mkdir(parents=True, exist_ok=True)

# 1. Cross-Model Comparison Bar Chart
plot_cross_model_comparison(eval_results, plots_output / 'cross_model_comparison.png', split_name='Held-Out Test Split')
print('Generated outputs/plots/cross_model_comparison.png')

# 2. Modality Ablation Bar Chart
ablation_summary = {
    'Video-Only (MobileNetV2)': eval_results.get('MobileNetV2 Fine-Tuned', {}),
    'Sensor-Only (InceptionTime)': eval_results.get('InceptionTime (Sensor-Only)', {}),
    'Multimodal Fusion': eval_results.get('Multimodal Fusion (Video+Sensor)', {}),
}
plot_modality_ablation(ablation_summary, plots_output / 'modality_ablation_comparison.png', split_name='Held-Out Test Split')
print('Generated outputs/plots/modality_ablation_comparison.png')

# Display publication gallery
gallery_plots = [
    ('Cross-Model Benchmark Comparison (All 5 Models)', 'outputs/plots/cross_model_comparison.png'),
    ('Modality Ablation (Video vs Sensor vs Multimodal Fusion)', 'outputs/plots/modality_ablation_comparison.png'),
    ('Hyperparameter Tuning Trials Comparison', 'outputs/plots/hyperparameter_trials.png'),
    ('MobileNetV2: Train vs Val Accuracy (Annotated 50 Epochs)', 'outputs/mobilenet_v2/plots/train_vs_val_accuracy_annotated_mobilenet_v2.png'),
    ('MobileNetV2: Train vs Val Loss (Annotated 50 Epochs)', 'outputs/mobilenet_v2/plots/train_vs_val_loss_annotated_mobilenet_v2.png'),
    ('MobileNetV2: Test Confusion Matrix', 'outputs/mobilenet_v2/plots/test_confusion_matrix_mobilenet_v2.png'),
    ('MobileNetV2: Standalone Per-Class F1-Score', 'outputs/mobilenet_v2/plots/test_f1_bars_mobilenet_v2.png'),
    ('MobileNetV2: Multiclass OvR ROC Curves', 'outputs/mobilenet_v2/plots/test_roc_ovr_mobilenet_v2.png'),
    ('MobileNetV2: Multiclass Precision-Recall Curves', 'outputs/mobilenet_v2/plots/test_pr_curves_mobilenet_v2.png'),
    ('Custom CNN: Train vs Val Accuracy (Annotated)', 'outputs/baseline_cnn/plots/train_vs_val_accuracy_annotated_cnn.png'),
    ('Multimodal Fusion: Test Confusion Matrix', 'outputs/multimodal_fusion/plots/test_confusion_matrix_mobilenet_v2.png'),
]

for title, p in gallery_plots:
    if Path(p).exists():
        print('\n' + '=' * 75)
        print(title)
        print('=' * 75)
        display(Image(filename=p))


## Section 14: Sample Multimodal Inference Demonstration
Runs standalone single-sample multimodal inference using the trained fusion model and prints predicted class probabilities.

In [ ]:
# Section 14: Sample Multimodal Inference Demonstration
sample_video = 'data/genuine_mmact/trimmed/video/cross_scene/trainval/subject16/cam1/scene1/session1/crouching.mp4'
sample_phone = 'data/genuine_mmact/trimmed/sensors/acc_phone_clip/subject16/scene1/session1/crouching.csv'
sample_watch = 'data/genuine_mmact/trimmed/sensors/acc_watch_clip/subject16/scene1/session1/crouching.csv'
sample_gyro  = 'data/genuine_mmact/trimmed/sensors/gyro_clip/subject16/scene1/session1/crouching.csv'

print('Running Multimodal Inference Demo on Test Sample...')
!python scripts/infer.py \
    --video {sample_video} \
    --sensor_phone {sample_phone} \
    --sensor_watch {sample_watch} \
    --sensor_gyro {sample_gyro} \
    --checkpoint outputs/multimodal_fusion/checkpoints/best_model_multimodal.pth


## Section 15: Consolidated Summary Table, Manifest, Runtime Report & 1-Click Packaging
Aggregates experiment manifests, runtime reports, and performance metrics into a consolidated summary table and packages `/kaggle/working/har_mmact_results.zip` for 1-click download.

In [ ]:
# Section 15: Consolidated Summary Table, Manifest, Runtime Report & 1-Click Packaging
import shutil
import pandas as pd
from pathlib import Path

print('=' * 80)
print(' CONSOLIDATED HAR_MMACT BENCHMARK SUMMARY TABLE')
print('=' * 80)
rows = []
for model_name, m in eval_results.items():
    rows.append({
        'Model Architecture': model_name,
        'Accuracy (%)': round(m.get('accuracy', 0.0) * 100.0, 2),
        'Macro F1': round(m.get('macro_f1', 0.0), 4),
        'Macro Precision': round(m.get('macro_precision', 0.0), 4),
        'Macro Recall': round(m.get('macro_recall', 0.0), 4),
    })
df_results = pd.DataFrame(rows)
print(df_results.to_markdown(index=False))
df_results.to_csv('outputs/benchmark_summary.csv', index=False)
print('Saved outputs/benchmark_summary.csv')

# Organize structured results into /kaggle/working/har_mmact_results
results_dir = Path('/kaggle/working/har_mmact_results')
results_dir.mkdir(parents=True, exist_ok=True)

for sub in ['baseline_cnn', 'tuned_cnn', 'mobilenet_v2', 'sensor_only', 'multimodal_fusion', 'tuning', 'plots']:
    src_p = Path('outputs') / sub
    if src_p.exists():
        shutil.copytree(src_p, results_dir / sub, dirs_exist_ok=True)

for f in ['benchmark_summary.csv']:
    src_f = Path('outputs') / f
    if src_f.exists():
        shutil.copy2(src_f, results_dir / f)

# Package into zip archive
archive_path = '/kaggle/working/har_mmact_results'
shutil.make_archive(archive_path, 'zip', results_dir)
zip_file = '/kaggle/working/har_mmact_results.zip'
print('\n' + '=' * 80)
print('SUCCESS: All benchmark experiments, checkpoints, and visualizations packaged!')
print(f'Download File: {zip_file}')
print('Available under Kaggle Output tab in the right-hand panel.')
print('=' * 78)


## Section 16: (Optional) Extended Benchmarks (100 and 150 Epochs)
This section is **strictly optional** and intended for extended GPU quota or overnight execution.
The required 50-epoch benchmark is fully satisfied by Sections 8 through 11. If GPU quota permits, you can toggle `RUN_EXTENDED = True` to train extended checkpoints.

In [ ]:
# Section 16: (Optional) Extended Benchmarks (100 and 150 Epochs)
# Set RUN_EXTENDED = True to enable extended training beyond the required 50-epoch baseline.
RUN_EXTENDED = False
EXTENDED_EPOCHS = 100  # Set to 100 or 150

if RUN_EXTENDED:
    print(f'=== Starting Optional Extended Training ({EXTENDED_EPOCHS} Epochs) ===')
    out_extended = f'outputs/multimodal_extended_{EXTENDED_EPOCHS}ep'
    Path(out_extended).mkdir(parents=True, exist_ok=True)
    !python scripts/train.py \
        --data_root data/genuine_mmact \
        --mode multimodal \
        --video_backbone mobilenet_v2 \
        --sensor_backbone inception_time \
        --epochs {EXTENDED_EPOCHS} \
        --min_epochs {EXTENDED_EPOCHS} \
        --batch_size 8 \
        --lr 0.0001 \
        --weight_decay 0.0005 \
        --split_strategy subject \
        --checkpoint_name best_model_multimodal_extended.pth \
        --output_dir {out_extended} \
        --cache_dir /kaggle/working/har_mmact_cache \
        --auto_resume \
        --skip_if_completed
else:
    print('Optional extended training skipped. Required 50-epoch benchmark is complete.')
